In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import pandas as pd
# Replace with your actual copied path
from google.colab import files
uploaded = files.upload()



Saving df_combined.csv to df_combined.csv
Saving local_efficiency_nodewise_thr03.csv to local_efficiency_nodewise_thr03.csv
Saving participation_nodewise_thr03.csv to participation_nodewise_thr03.csv
Saving clustering_nodewise_thr03.csv to clustering_nodewise_thr03.csv
Saving measures_multi_threshold.csv to measures_multi_threshold.csv
Saving Global_metrics_threshold03.csv to Global_metrics_threshold03.csv
Saving betweenness_nodewise_thr03.csv to betweenness_nodewise_thr03.csv


In [6]:
# Directory (check file icon on the left and right click on the file fro file path)
# Global
df_global = pd.read_csv('/content/drive/MyDrive/Graph Theory Machine Learning Files/Global_metrics_threshold03.csv')
# MultiThreshohold Global
df_multi = pd.read_csv('/content/drive/MyDrive/Graph Theory Machine Learning Files/measures_multi_threshold.csv')
# Local
df_local_p = pd.read_csv('/content/participation_nodewise_thr03.csv')
df_local_e = pd.read_csv('/content/local_efficiency_nodewise_thr03.csv')
df_local_c = pd.read_csv('/content/clustering_nodewise_thr03.csv')
df_local_b = pd.read_csv('/content/betweenness_nodewise_thr03.csv')

#Combined local and global
df_combined = pd.read_csv('/content/df_combined.csv')

# BASE MODEL (No adjustements)

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
import pandas as pd
from sklearn.impute import SimpleImputer # Import SimpleImputer

df_local = pd.concat([df_local_p, df_local_e, df_local_c, df_local_b], axis=1) # axis = 1 Ensures horizontal concatenation
df_local.to_csv('df_local.csv', index = False)
files.download('df_local.csv')
#print(df_local)


# Combining Local and Global Varibales

# Combine local and global
# df_combined = pd.concat([df_global, df_local], axis=1)
# df_combined = df_combined.loc[:,~df_combined.columns.duplicated()].copy()

# Ensure df_combined has unique columns, keeping the first occurrence
df_combined = df_combined.loc[:,~df_combined.columns.duplicated()].copy()

#df_combined.to_csv('df_combined.csv', index = False)
#files.download('df_combined.csv')

# Converts CN and AD labels to 0 and 1
y = df_combined["diagnosis"].map({"CN": 0, "AD": 1}).astype(int) # target variable (label)

# Drop primary 'subject_id' and 'diagnosis' columns first
X = df_combined.drop(columns=["subject_id", "diagnosis", "threshold_value" ]).copy() # independent variable (predictor)
display(X.head())
# Identify and drop any remaining object-type (string or string similar types) columns that are not the intended 'sex' column
object_cols_to_drop_additionally = [col for col in X.select_dtypes(include=['object']).columns if col != 'sex']
X = X.drop(columns=object_cols_to_drop_additionally, errors='ignore')

# Convert categorical 'sex' column to numerical using one-hot encoding, if it still exists
if 'sex' in X.columns:
    X = pd.get_dummies(X, columns=['sex'], drop_first=True) # drop_first=True avoids multicollinearity

# Print information about X to diagnose non-numeric columns
print("X DataFrame info before model fit:")
print(X.info())
print("Non-numeric columns in X:")
print(X.select_dtypes(include=['object']).columns)

# Impute missing values with the mean
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)


#files.download('df_combined.csv')
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42) #80/20 split
model = SVC(kernel = "linear", C = 5) #Changing C does nothing to the model, nonlinear kernel because number of features is small and number of observations big
# model = SVC(kernel = "rbf", C = 5) # nonlinear kernel gets accuracy of 0.625
model = model.fit(X_train, y_train)
model.score(X_test, y_test)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,sex,network_density,mean_clustering_coefficient,global_efficiency,modularity_Q,mean_participation_coefficient,mean_betweenness_centrality,small_world_sigma,small_world_gamma,small_world_lambda,...,RM-PFCdl_L.3,RM-PMCm_L.3,RM-PMCdl_L.3,TM-F_L.3,TM-T_L.3,TM-OP_L.3,BG-Cd_L.3,BG-Pu_L.3,BG-Pa_L.3,BG-Acc_L.3
0,M,0.200658,0.211975,0.265494,0.328885,0.512549,95.041667,1.455033,1.525831,1.048657,...,74.0,34.0,2.0,58.0,144.0,0.0,34.0,210.0,0.0,4.0
1,M,0.240570,0.230207,0.287986,0.351212,0.485758,84.520833,1.561112,1.606816,1.029276,...,136.0,112.0,140.0,132.0,188.0,16.0,142.0,194.0,0.0,24.0
2,M,0.157456,0.166125,0.240326,0.376496,0.582348,102.958333,1.815091,1.893201,1.043034,...,12.0,168.0,50.0,94.0,6.0,12.0,2.0,94.0,14.0,74.0
3,M,0.249342,0.223333,0.292951,0.278111,0.488952,80.625000,1.365717,1.396937,1.022860,...,62.0,60.0,30.0,88.0,68.0,54.0,16.0,108.0,14.0,34.0
4,M,0.211404,0.210395,0.272057,0.359048,0.495980,89.895833,1.578448,1.619104,1.025757,...,28.0,38.0,44.0,62.0,44.0,10.0,106.0,20.0,0.0,10.0


X DataFrame info before model fit:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Columns: 402 entries, network_density to sex_M
dtypes: bool(1), float64(401)
memory usage: 125.5 KB
None
Non-numeric columns in X:
Index([], dtype='object')


0.75

# CROSS VALIDATION INCLUDED

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, KFold
import pandas as pd
from sklearn.impute import SimpleImputer # Import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline # Import Pipeline
from sklearn.preprocessing import StandardScaler # Import StandardScaler


df_local = pd.concat([df_local_p, df_local_e, df_local_c, df_local_b], axis=1) # axis = 1 Ensures horizontal concatenation

# Ensure df_combined has unique columns, keeping the first occurrence
df_combined = df_combined.loc[:,~df_combined.columns.duplicated()].copy()

# Converts CN and AD labels to 0 and 1
y = df_combined["diagnosis"].map({"CN": 0, "AD": 1}).astype(int) # target variable (label)

# Drop primary 'subject_id' and 'diagnosis' columns first
X = df_combined.drop(columns=["subject_id", "diagnosis", "threshold_value" ]).copy() # independent variable (predictor)
display(X.head())
# Identify and drop any remaining object-type (string or string similar types) columns that are not the intended 'sex' column
object_cols_to_drop_additionally = [col for col in X.select_dtypes(include=['object']).columns if col != 'sex']
X = X.drop(columns=object_cols_to_drop_additionally, errors='ignore')

# Convert categorical 'sex' column to numerical using one-hot encoding, if it still exists
if 'sex' in X.columns:
    X = pd.get_dummies(X, columns=['sex'], drop_first=True) # drop_first=True avoids multicollinearity

# Print information about X to diagnose non-numeric columns
print("X DataFrame info before model fit:")
print(X.info())
print("Non-numeric columns in X:")
print(X.select_dtypes(include=['object']).columns)

# Impute missing values with the mean
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

# Define the model
model = SVC(C = 5, kernel = "linear") # Added kernel explicitly for consistency

# Re-add SimpleImputer to the pipeline to handle NaNs within cross-validation
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler()),
    ('svc', SVC(C=10, kernel="rbf"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=cv) # Now pass X, as imputer is in the pipe

print(scores)
print(scores.mean())

# Perform cross-validation with the model directly on X_imputed (for comparison)
kf = KFold(n_splits=5, shuffle=True, random_state=42) # 5-fold cross-validation
cv_scores = cross_val_score(model, X_imputed, y, cv=kf)

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean cross-validation accuracy: {cv_scores.mean():.4f}")

,sex,network_density,mean_clustering_coefficient,global_efficiency,modularity_Q,mean_participation_coefficient,mean_betweenness_centrality,small_world_sigma,small_world_gamma,small_world_lambda,...,RM-PFCdl_L.3,RM-PMCm_L.3,RM-PMCdl_L.3,TM-F_L.3,TM-T_L.3,TM-OP_L.3,BG-Cd_L.3,BG-Pu_L.3,BG-Pa_L.3,BG-Acc_L.3
0,M,0.200658,0.211975,0.265494,0.328885,0.512549,95.041667,1.455033,1.525831,1.048657,...,74.0,34.0,2.0,58.0,144.0,0.0,34.0,210.0,0.0,4.0
1,M,0.240570,0.230207,0.287986,0.351212,0.485758,84.520833,1.561112,1.606816,1.029276,...,136.0,112.0,140.0,132.0,188.0,16.0,142.0,194.0,0.0,24.0
2,M,0.157456,0.166125,0.240326,0.376496,0.582348,102.958333,1.815091,1.893201,1.043034,...,12.0,168.0,50.0,94.0,6.0,12.0,2.0,94.0,14.0,74.0
3,M,0.249342,0.223333,0.292951,0.278111,0.488952,80.625000,1.365717,1.396937,1.022860,...,62.0,60.0,30.0,88.0,68.0,54.0,16.0,108.0,14.0,34.0
4,M,0.211404,0.210395,0.272057,0.359048,0.495980,89.895833,1.578448,1.619104,1.025757,...,28.0,38.0,44.0,62.0,44.0,10.0,106.0,20.0,0.0,10.0


X DataFrame info before model fit:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Columns: 402 entries, network_density to sex_M
dtypes: bool(1), float64(401)
memory usage: 125.5 KB
None
Non-numeric columns in X:
Index([], dtype='object')
[0.75  0.5   0.5   0.625 0.75 ]
0.625
Cross-validation scores: [0.75  0.625 0.625 0.25  0.25 ]
Mean cross-validation accuracy: 0.5000


# HYPERPARAMETER TUNING

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
import pandas as pd
from sklearn.impute import SimpleImputer # Import SimpleImputer

df_local = pd.concat([df_local_p, df_local_e, df_local_c, df_local_b], axis=1) # axis = 1 Ensures horizontal concatenation
df_local.to_csv('df_local.csv', index = False)
files.download('df_local.csv')
#print(df_local)


# Combining Local and Global Varibales

# Combine local and global
# df_combined = pd.concat([df_global, df_local], axis=1)
# df_combined = df_combined.loc[:,~df_combined.columns.duplicated()].copy()

# Ensure df_combined has unique columns, keeping the first occurrence
df_combined = df_combined.loc[:,~df_combined.columns.duplicated()].copy()

#df_combined.to_csv('df_combined.csv', index = False)
#files.download('df_combined.csv')

# Converts CN and AD labels to 0 and 1
y = df_combined["diagnosis"].map({"CN": 0, "AD": 1}).astype(int) # target variable (label)

# Drop primary 'subject_id' and 'diagnosis' columns first
X = df_combined.drop(columns=["subject_id", "diagnosis", "threshold_value" ]).copy() # independent variable (predictor)
display(X.head())
# Identify and drop any remaining object-type (string or string similar types) columns that are not the intended 'sex' column
object_cols_to_drop_additionally = [col for col in X.select_dtypes(include=['object']).columns if col != 'sex']
X = X.drop(columns=object_cols_to_drop_additionally, errors='ignore')

# Convert categorical 'sex' column to numerical using one-hot encoding, if it still exists
if 'sex' in X.columns:
    X = pd.get_dummies(X, columns=['sex'], drop_first=True) # drop_first=True avoids multicollinearity

# Print information about X to diagnose non-numeric columns
print("X DataFrame info before model fit:")
print(X.info())
print("Non-numeric columns in X:")
print(X.select_dtypes(include=['object']).columns)

# Impute missing values with the mean
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)


#files.download('df_combined.csv')
X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42) #80/20 split
model = SVC(kernel = "linear", C = 5) #Changing C does nothing to the model, nonlinear kernel because number of features is small and number of observations big
# model = SVC(kernel = "rbf", C = 5) # nonlinear kernel gets accuracy of 0.625
model = model.fit(X_train, y_train)
model.score(X_test, y_test)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,sex,network_density,mean_clustering_coefficient,global_efficiency,modularity_Q,mean_participation_coefficient,mean_betweenness_centrality,small_world_sigma,small_world_gamma,small_world_lambda,...,RM-PFCdl_L.3,RM-PMCm_L.3,RM-PMCdl_L.3,TM-F_L.3,TM-T_L.3,TM-OP_L.3,BG-Cd_L.3,BG-Pu_L.3,BG-Pa_L.3,BG-Acc_L.3
0,M,0.200658,0.211975,0.265494,0.328885,0.512549,95.041667,1.455033,1.525831,1.048657,...,74.0,34.0,2.0,58.0,144.0,0.0,34.0,210.0,0.0,4.0
1,M,0.240570,0.230207,0.287986,0.351212,0.485758,84.520833,1.561112,1.606816,1.029276,...,136.0,112.0,140.0,132.0,188.0,16.0,142.0,194.0,0.0,24.0
2,M,0.157456,0.166125,0.240326,0.376496,0.582348,102.958333,1.815091,1.893201,1.043034,...,12.0,168.0,50.0,94.0,6.0,12.0,2.0,94.0,14.0,74.0
3,M,0.249342,0.223333,0.292951,0.278111,0.488952,80.625000,1.365717,1.396937,1.022860,...,62.0,60.0,30.0,88.0,68.0,54.0,16.0,108.0,14.0,34.0
4,M,0.211404,0.210395,0.272057,0.359048,0.495980,89.895833,1.578448,1.619104,1.025757,...,28.0,38.0,44.0,62.0,44.0,10.0,106.0,20.0,0.0,10.0


X DataFrame info before model fit:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Columns: 402 entries, network_density to sex_M
dtypes: bool(1), float64(401)
memory usage: 125.5 KB
None
Non-numeric columns in X:
Index([], dtype='object')


0.75

In [10]:
print("X head:")
display(X.head())
print("X shape:", X.shape)

print("\ny head:")
display(y.head())
print("y shape:", y.shape)

X head:


,network_density,mean_clustering_coefficient,global_efficiency,modularity_Q,mean_participation_coefficient,mean_betweenness_centrality,small_world_sigma,small_world_gamma,small_world_lambda,threshold_value.1,...,RM-PMCm_L.3,RM-PMCdl_L.3,TM-F_L.3,TM-T_L.3,TM-OP_L.3,BG-Cd_L.3,BG-Pu_L.3,BG-Pa_L.3,BG-Acc_L.3,sex_M
0,0.200658,0.211975,0.265494,0.328885,0.512549,95.041667,1.455033,1.525831,1.048657,0.3,...,34.0,2.0,58.0,144.0,0.0,34.0,210.0,0.0,4.0,True
1,0.240570,0.230207,0.287986,0.351212,0.485758,84.520833,1.561112,1.606816,1.029276,0.3,...,112.0,140.0,132.0,188.0,16.0,142.0,194.0,0.0,24.0,True
2,0.157456,0.166125,0.240326,0.376496,0.582348,102.958333,1.815091,1.893201,1.043034,0.3,...,168.0,50.0,94.0,6.0,12.0,2.0,94.0,14.0,74.0,True
3,0.249342,0.223333,0.292951,0.278111,0.488952,80.625000,1.365717,1.396937,1.022860,0.3,...,60.0,30.0,88.0,68.0,54.0,16.0,108.0,14.0,34.0,True
4,0.211404,0.210395,0.272057,0.359048,0.495980,89.895833,1.578448,1.619104,1.025757,0.3,...,38.0,44.0,62.0,44.0,10.0,106.0,20.0,0.0,10.0,True


X shape: (40, 402)

y head:


,diagnosis
0,1
1,1
2,1
3,1
4,1


y shape: (40,)


Binary Classifier (1 for AD, 0 for CN), SVM
* Hyperparameters in sklearn: gamma, C, degree, coef0, shrinking, probability

In [11]:
param_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__kernel': ['linear', 'rbf', 'poly', 'sigmoid']
}
print("Hyperparameter grid defined:")
print(param_grid)

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Create a Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42))
])

# Initialize GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the data
grid_search.fit(X_imputed, y)

print("GridSearchCV completed.")

print("Best hyperparameters found: ", grid_search.best_params_)
print("Best cross-validation accuracy: {:.4f}".format(grid_search.best_score_))

Hyperparameter grid defined:
{'svc__C': [0.1, 1, 10, 100], 'svc__kernel': ['linear', 'rbf', 'poly', 'sigmoid']}
Fitting 5 folds for each of 16 candidates, totalling 80 fits
GridSearchCV completed.
Best hyperparameters found:  {'svc__C': 10, 'svc__kernel': 'rbf'}
Best cross-validation accuracy: 0.7250


In [14]:
import numpy as np
from sklearn.linear_model import LinearRegression
model = LinearRegression().fit(X_imputed, y)
r_sq = model.score(X_imputed, y)
print(f"R-squared: {r_sq}")
print(f"Intercept (b0): {model.intercept_}")
print(f"Slope (b1): {model.coef_}")
y_pred = model.predict(X_imputed)
print(f"Predicted responses: {y_pred}")


R-squared: 1.0
Intercept (b0): 0.955839472004556
Slope (b1): [ 6.79889640e-08 -2.80414008e-07  2.75959360e-08 -2.80528018e-07
 -4.58406375e-07 -4.94792469e-05  2.60815514e-07  1.22916517e-07
  3.15616499e-08 -3.25260652e-19  6.79832151e-08  6.29224456e-07
  8.11686697e-07  1.08126234e-06 -1.23339667e-06  9.70367045e-07
 -3.00950768e-07  2.11883417e-07  2.71020751e-06  1.47632974e-06
 -9.07782510e-07 -1.07724464e-06 -1.29370488e-06 -1.65364015e-06
 -1.86182026e-06  2.46422285e-06  1.77917113e-06 -3.10949886e-07
 -1.14243912e-06  1.15548965e-06 -5.29989392e-07  1.98545488e-06
 -1.36882347e-07 -8.31479831e-07  2.06854022e-06 -2.22293839e-06
  1.38447344e-07  1.21066898e-07 -1.95346953e-06 -3.09820506e-06
 -1.20903998e-06  3.09998234e-07 -1.81532701e-06 -1.63592473e-06
 -1.26166979e-06 -3.96747525e-07 -8.21427774e-07 -4.08067291e-07
 -3.17504273e-07 -8.29113373e-07 -7.05612771e-07 -1.33732185e-06
 -9.46025160e-07 -3.09684722e-06 -1.69443504e-07 -2.96593045e-06
  1.99019097e-07 -9.39555434e